# Module 7.1: Distributed Tracing with OpenTelemetry

## Introduction & Learning Objectives

### The Problem

You have Prometheus metrics showing **P95 latency: 850ms**. Everything looks healthy.

Then a user reports: *"My query took 4.2 seconds!"*

Aggregate metrics hide individual request failures. You need **request-level visibility** through the entire RAG pipeline:
- Retrieval (Pinecone query)
- Reranking (Cohere API)
- Generation (OpenAI GPT-4)

**Distributed tracing** provides "tracking numbers" (trace IDs) that follow requests through multiple services, showing exactly where time is spent.

### Learning Objectives

By the end of this module, you will:

1. ✅ Initialize OpenTelemetry tracer with production-ready `BatchSpanProcessor`
2. ✅ Auto-instrument FastAPI for HTTP request tracing
3. ✅ Add manual spans for RAG pipeline stages (retrieval, reranking, generation)
4. ✅ Visualize traces in Jaeger UI
5. ✅ Correlate traces with logs using trace IDs
6. ✅ Configure sampling for production (10-50% to reduce overhead)
7. ✅ Understand when **NOT** to use tracing (and what to use instead)

### Prerequisites

This assumes completion of **Level 1 M2.3** (Prometheus/Grafana monitoring):
- ✅ Prometheus metrics collection
- ✅ Grafana dashboards for P50/P95/P99 latency
- ✅ Structured JSON logging
- ✅ Basic health checks

**Gap:** Can't debug individual slow requests—only see aggregate statistics.

## Section 2: Dependencies & Setup

Install OpenTelemetry SDK, exporters, and instrumentation packages.

In [ ]:
# Install dependencies (if needed)
# !pip install -r requirements.txt

# Import required modules
import sys
import json
from pathlib import Path

# Verify installations
try:
    from opentelemetry import trace
    from opentelemetry.sdk.trace import TracerProvider
    from config import tracing_config, validate_config
    import l2_m7_distributed_tracing_opentelemetry as tracing_module
    
    print("✅ All dependencies installed successfully!")
    print(f"\nConfiguration:")
    print(f"  Service: {tracing_config.SERVICE_NAME}")
    print(f"  Environment: {tracing_config.ENVIRONMENT}")
    print(f"  OTLP Endpoint: {tracing_config.OTLP_ENDPOINT}")
    print(f"  Sampling Rate: {tracing_config.SAMPLING_RATE * 100}%")
    
    # Validate config
    validation = validate_config()
    print(f"\nValidation:")
    for key, value in validation.items():
        status = "✅" if value else "⚠️"
        print(f"  {status} {key}: {value}")
        
except ImportError as e:
    print(f"❌ Missing dependency: {e}")
    print("Run: pip install -r requirements.txt")

# Expected:
# ✅ All dependencies installed
# Configuration shows service name, endpoint, sampling rate
# Validation shows tracing_enabled=True (jaeger_reachable may be False if not started)

## Section 3: Theory Foundation

### Three Pillars of Observability

Distributed tracing is the **third pillar** of observability, complementing metrics and logs:

| Pillar | Purpose | Example Question |
|--------|---------|------------------|
| **Metrics** | Aggregate time-series data | *"What is P95 latency?"* |
| **Logs** | Event text records | *"What errors happened?"* |
| **Traces** | Request paths & timing | *"Why was THIS request slow?"* |

### Key Concepts

1. **Trace IDs** = Tracking numbers for requests
   - Unique identifier following a request through all services
   - Format: 32-character hex string (e.g., `a1b2c3d4e5f6...`)

2. **Spans** = Segments of work
   - Parent spans contain child spans (hierarchical)
   - Each span has: name, start time, duration, attributes, status

3. **Context Propagation** = Trace ID flows through function calls
   - Automatic in synchronous code
   - Requires explicit handling in threading/async

4. **Export** = Spans sent to Jaeger for visualization
   - Batched every 5 seconds (configurable)
   - OTLP protocol over gRPC (port 4317)

### Critical Note

**Tracing does NOT replace metrics!** They work together:
- Metrics: Aggregate health (P95 latency trend over time)
- Traces: Individual debugging (why did request X take 4.2 seconds?)

At scale, metrics are cheaper (keep 1+ years) vs traces (7-30 days typical).

In [ ]:
# Visualize example trace structure
print("Example Trace Structure:")
print("=" * 60)
print()
print("Trace ID: a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6")
print()
print("rag.query (1200ms) ◄── Parent span")
print("  │")
print("  ├─ pinecone.retrieve (200ms)")
print("  │    ├─ embedding.generate (50ms)")
print("  │    └─ pinecone.query (150ms)")
print("  │")
print("  ├─ reranking.process (200ms)")
print("  │")
print("  └─ llm.generate (800ms)")
print()
print("Total: 1200ms across 6 spans")
print()
print("Attributes captured:")
print("  • question: 'What are GDPR requirements?'")
print("  • top_k: 10, top_n: 5")
print("  • results.count: 10")
print("  • llm.tokens: 2650")
print("  • llm.cost_usd: 0.000084")

# Expected:
# ASCII diagram showing hierarchical span structure
# Parent span contains nested child spans
# Total time = sum of sequential operations (not parallel)

## Section 4: Step 1 - Initialize Tracer

Configure OpenTelemetry with **BatchSpanProcessor** for production-ready async export.

### Why BatchSpanProcessor?

| Processor | Latency Overhead | Use Case |
|-----------|------------------|----------|
| `SimpleSpanProcessor` | **50-100ms per request** | Development only |
| `BatchSpanProcessor` | **1-2ms per request** | Production (recommended) |

**BatchSpanProcessor** buffers spans and exports in batches asynchronously:
- Buffer: 2048 spans max
- Batch size: 512 spans per export
- Schedule: Export every 5 seconds

Trade-off: 5-second delay before traces appear in Jaeger vs blocking request processing.

In [ ]:
# Initialize tracer with production-ready configuration
from l2_m7_distributed_tracing_opentelemetry import setup_tracing
from config import tracing_config

tracer = setup_tracing(
    service_name=tracing_config.SERVICE_NAME,
    environment=tracing_config.ENVIRONMENT,
    version=tracing_config.SERVICE_VERSION,
    otlp_endpoint=tracing_config.OTLP_ENDPOINT,
    sampling_rate=tracing_config.SAMPLING_RATE,
    max_queue_size=tracing_config.MAX_QUEUE_SIZE,
    max_export_batch_size=tracing_config.MAX_EXPORT_BATCH_SIZE,
    schedule_delay_millis=tracing_config.SCHEDULE_DELAY_MILLIS
)

print("✅ Tracer initialized successfully!")
print(f"\nConfiguration:")
print(f"  Service: {tracing_config.SERVICE_NAME}")
print(f"  Environment: {tracing_config.ENVIRONMENT}")
print(f"  OTLP Endpoint: {tracing_config.OTLP_ENDPOINT}")
print(f"  Sampling: {tracing_config.SAMPLING_RATE * 100}%")
print(f"  Batch Config: buffer={tracing_config.MAX_QUEUE_SIZE}, ")
print(f"                batch={tracing_config.MAX_EXPORT_BATCH_SIZE}, ")
print(f"                delay={tracing_config.SCHEDULE_DELAY_MILLIS}ms")
print(f"\n⚠️  Note: Traces export every {tracing_config.SCHEDULE_DELAY_MILLIS/1000}s (async)")

# Expected:
# ✅ Tracer initialized (works even if Jaeger not running - graceful degradation)
# Shows service name, sampling rate, batch configuration

## Section 5: Step 2 - Run Jaeger

Jaeger is the trace visualization backend. Run locally via Docker.

### Jaeger Ports

| Port | Purpose |
|------|---------|
| **16686** | Jaeger UI (web interface) |
| **4317** | OTLP gRPC collector (OpenTelemetry) |
| **4318** | OTLP HTTP collector (alternative) |

### Docker Command

```bash
docker run -d --name jaeger \
  -e COLLECTOR_OTLP_ENABLED=true \
  -p 16686:16686 \
  -p 4317:4317 \
  -p 4318:4318 \
  jaegertracing/all-in-one:1.51
```

**Production:** Add TTL and memory limits:
```bash
docker run -d --name jaeger \
  -e COLLECTOR_OTLP_ENABLED=true \
  -e BADGER_TTL=168h \
  -e BADGER_MAINTENANCE_INTERVAL=1h \
  --memory=2g \
  -p 16686:16686 -p 4317:4317 \
  jaegertracing/all-in-one:1.51
```

TTL=168h = 7 days retention (prevents disk overflow).

In [ ]:
# Check if Jaeger is running
import socket
from config import validate_config

print("Checking Jaeger status...")
print("=" * 60)

validation = validate_config()

if validation.get("jaeger_reachable"):
    print("✅ Jaeger is running!")
    print(f"\n   UI: {tracing_config.JAEGER_UI_URL}")
    print(f"   OTLP gRPC: {tracing_config.OTLP_ENDPOINT}")
else:
    print("⚠️  Jaeger not reachable")
    print("\nStart Jaeger with:")
    print("  docker run -d --name jaeger \\")
    print("    -e COLLECTOR_OTLP_ENABLED=true \\")
    print("    -p 16686:16686 -p 4317:4317 -p 4318:4318 \\")
    print("    jaegertracing/all-in-one:1.51")
    print("\n  Then check: http://localhost:16686")
    print("\n⚠️  Application will continue without tracing (graceful degradation)")

# Expected:
# ✅ if Jaeger running, shows UI URL
# ⚠️  if not running, shows Docker command (app still works without tracing)

## Section 6: Step 3 - Instrument RAG Pipeline

Add manual spans for retrieval, reranking, and generation stages.

### Span Creation Pattern

```python
with tracer.start_as_current_span("operation_name", attributes={...}) as span:
    # Do work
    result = expensive_operation()
    
    # Add result attributes
    span.set_attribute("result_count", len(result))
    
    # Automatic: duration, start/end times
    # On exception: automatically records error status
```

### Key Attributes to Capture

**Retrieval:**
- `question` (truncated to 100 chars)
- `top_k`, `results.count`
- `db.system` (e.g., "pinecone")

**Reranking:**
- `input_count`, `output_count`
- `model` (e.g., "cohere-rerank-v3")
- `api.status_code`

**Generation:**
- `model` (e.g., "gpt-4")
- `llm.prompt_tokens`, `llm.completion_tokens`
- `llm.cost_usd`

In [ ]:
# Run RAG query with full tracing instrumentation
from l2_m7_distributed_tracing_opentelemetry import (
    process_rag_query,
    get_trace_context
)
import time

print("Running RAG query with distributed tracing...")
print("=" * 60)

# Process query
result = process_rag_query(
    tracer=tracer,
    question="What are GDPR compliance requirements for data retention?",
    top_k=10,
    top_n=5,
    model="gpt-4"
)

# Get trace context for correlation
trace_ctx = get_trace_context()

print(f"\n✅ Query completed!")
print(f"\nResult:")
print(f"  Response: {result['response'][:80]}...")
print(f"  Time: {result['total_time_ms']:.0f}ms")
print(f"  Documents: {result['retrieved_count']} → {result['reranked_count']}")
print(f"  Tokens: {result['tokens']} (${result['cost_usd']:.6f})")

print(f"\nTrace Context:")
print(f"  Trace ID: {trace_ctx['trace_id']}")
print(f"  Span ID: {trace_ctx['span_id']}")

if validation.get("jaeger_reachable"):
    jaeger_url = f"{tracing_config.JAEGER_UI_URL}/trace/{trace_ctx['trace_id']}"
    print(f"\n🔍 View in Jaeger:")
    print(f"   {jaeger_url}")
    print(f"\n   (Wait 5 seconds for batch export)")
else:
    print(f"\n⚠️  Jaeger not running - trace not exported")

# Expected:
# ✅ Query completes in ~1200ms
# Shows retrieved → reranked document counts
# Displays trace_id for Jaeger lookup
# If Jaeger running, provides direct UI link

## Section 7: Reality Check (TVH Framework v2.0)

### What This DOESN'T Do

❌ **1. Replace Metrics/Logs**

Tracing is **10-100x more expensive** than metrics:
- Metrics: Keep 1+ years, cheap storage ($5-10/month)
- Traces: 7-30 days typical, expensive at scale ($50-500/month at 10K+ req/day)

**Example:** To know "What was P95 latency last month?" → Use **metrics**, not traces.

❌ **2. Work Without Overhead**

Adds **10-20ms per request** at 100% sampling:
- 5-10ms span creation
- 5-10ms serialization
- At 1000 req/day: 10-20 hours cumulative latency added daily
- For sub-100ms services: 10-20% overhead

❌ **3. Scale for Free**

At 10K requests/day with 100% sampling:
- Generates 50-100K spans/day
- 5KB per span = 250-500MB/day
- Over 30 days = **7-15GB storage**
- Jaeger storage fills, queries slow (10+ seconds)

### Trade-offs Accepted

| Aspect | Cost |
|--------|------|
| **Complexity** | 4 dependencies, 1 service (Jaeger), ~150 lines code |
| **Performance** | 10-20ms overhead (1-2ms at 10% sampling) |
| **Monthly Cost** | $0 (<2K req/day), $20-50 (24K req/day), $200-500 (240K+ req/day) |
| **Operational** | Monitoring disk, queries, memory, regular cleanups |

### When This Breaks

At **100K+ requests/day**:
- Self-hosted Jaeger: 150-300GB/month storage
- Queries slow: 20+ seconds
- 1% sampling misses 99% of requests
- Need columnar storage (Clickhouse, Tempo) or managed APM

**Bottom line:** Self-hosted OpenTelemetry + Jaeger works for **1K-10K req/day with 10-50% sampling**. At 100K+ req/day or 90+ day retention, upgrade to managed APM.

In [ ]:
# Calculate storage and cost for different traffic levels
print("Tracing Cost Calculator")
print("=" * 60)

traffic_scenarios = [
    {"name": "Low Traffic", "req_per_day": 2400, "sampling": 1.0, "retention_days": 7},
    {"name": "Medium Traffic", "req_per_day": 24000, "sampling": 0.2, "retention_days": 7},
    {"name": "High Traffic", "req_per_day": 240000, "sampling": 0.1, "retention_days": 30}
]

for scenario in traffic_scenarios:
    req_per_day = scenario["req_per_day"]
    sampling = scenario["sampling"]
    retention = scenario["retention_days"]
    
    # Calculate traces
    traces_per_day = req_per_day * sampling
    spans_per_trace = 6  # retrieval, embedding, query, rerank, llm, parent
    spans_per_day = traces_per_day * spans_per_trace
    
    # Storage (5KB per span)
    storage_mb_per_day = (spans_per_day * 5) / 1024
    storage_total_mb = storage_mb_per_day * retention
    storage_total_gb = storage_total_mb / 1024
    
    # Cost estimate
    if req_per_day < 5000:
        cost = 0
    elif req_per_day < 50000:
        cost = 20 + (storage_total_gb * 2)
    else:
        cost = 200 + (storage_total_gb * 5)
    
    print(f"\n{scenario['name']}: {req_per_day:,} req/day, {sampling*100}% sampling")
    print(f"  Traces/day: {traces_per_day:,.0f}")
    print(f"  Storage ({retention}d): {storage_total_gb:.1f} GB")
    print(f"  Est. cost: ${cost:.0f}/month")

# Expected:
# Low Traffic: $0/month (400MB storage, fits in free tier)
# Medium Traffic: $20-50/month (need Elasticsearch)
# High Traffic: $200-500/month (need Tempo or managed APM)

## Section 8: When NOT to Use (Anti-Patterns)

### ❌ Anti-Pattern 1: MVP Phase (<100 requests/day)

**Why it fails:**
- Setup time: 8+ hours
- Debugging time saved: <1 hour/month
- Architecture changes frequently
- No traffic volume justifies overhead

**Use instead:** Structured logging. Upgrade to tracing at 500+ req/day.

---

### ❌ Anti-Pattern 2: Single-Service Monolith

**Why it fails:**
- Tracing designed for **distributed systems** (multiple services)
- Single-service doesn't benefit from trace propagation
- Context already in memory

**Use instead:** Python profilers (`py-spy`, `cProfile`) for code-level analysis.

---

### ❌ Anti-Pattern 3: Cost-Constrained (<$50/month total)

**Why it fails:**
- Self-hosted Jaeger costs $20-50/month (compute + storage)
- Time cost: 8 hours setup + 2 hours/month maintenance
- Exceeds logging cost ($5-10/month)

**Use instead:** Cloud provider logging until budget allows.

---

### Summary: When to Avoid

| Condition | Why Avoid | Alternative |
|-----------|-----------|-------------|
| <100 req/day | Setup time > debugging time | Structured logging |
| Single service | No distributed context | Python profilers |
| Budget <$50/mo | Cost exceeds value | Cloud logging |
| MVP phase | Architecture changes | Wait for stability |
| No DevOps | Operational burden | Managed APM or wait |

**Use tracing if:** >500 req/day, distributed system, budget >$50/month, stable architecture, post-MVP.

## Section 9: Common Failures & Fixes

### Failure 1: Missing Trace Context Propagation

**Symptom:** Two separate traces instead of one connected trace

**Cause:** Python threading doesn't auto-propagate OpenTelemetry context

**Fix:**
```python
from opentelemetry import context

current_context = context.get_current()
thread = threading.Thread(
    target=my_function,
    args=(data, current_context)
)
```

**Prevention:** Use `asyncio` instead of threading, install instrumentation packages

---

### Failure 2: High Overhead (10-20% Latency)

**Symptom:** P95 latency jumps 850ms → 1020ms (+20%)

**Cause:** 100% sampling with fine-grained spans (per loop iteration)

**Fix:** Reduce sampling to 10%, use coarser spans

**Result:** 850ms → 862ms (+1.4% acceptable)

---

### Failure 3: Sampling Misses Critical Traces

**Symptom:** Search for slow request's trace_id → NOT FOUND

**Cause:** Head-based sampling decides before request completes

**Fix:** Use tail-based sampling or force-sample errors:
```python
if is_error or latency_ms > 2000:
    # Always sample slow/error requests
    with tracer.start_as_current_span(\"critical.op\", sampling_probability=1.0):
        process()
```

---

### Failure 4: Jaeger Storage Overflow

**Symptom:** Jaeger UI hangs (30+ second queries), OOM errors

**Cause:** 10K traces/day × 30 days = 300K traces = 1.5GB, no cleanup

**Fix:** Configure TTL:
```bash
docker run ... -e BADGER_TTL=168h -e BADGER_MAINTENANCE_INTERVAL=1h
```

**Better:** Use Elasticsearch backend for production

---

### Failure 5: Incomplete Span Coverage

**Symptom:** Total time (1200ms) > sum of spans (780ms), missing 420ms

**Cause:** HTTP client library not instrumented

**Fix:** Install instrumentation or add manual spans:
```bash
pip install opentelemetry-instrumentation-httpx
```"

In [ ]:
# Load failure scenarios from example data
with open("example_data.json", "r") as f:
    example_data = json.load(f)

print("Common Failure Scenarios")
print("=" * 60)

failures = example_data["failure_scenarios"]

for name, failure in failures.items():
    print(f"\n{name.replace('_', ' ').title()}")
    print(f"  Symptom: {failure['symptom']}")
    print(f"  Fix: {failure['fix']}")

# Expected:
# Lists 5 common failures with symptoms and fixes
# Missing context propagation, high overhead, storage overflow, etc.

## Section 10: Decision Card

### ✅ BENEFIT

**Request-level visibility** into retrieval → reranking → generation with sub-millisecond precision. Debug slow requests by seeing exactly where time went. Correlate traces with metrics and logs using trace IDs.

**Example:** User reports 4.2s query. Trace shows: retrieval 200ms, reranking 200ms, **LLM 3800ms** (bottleneck identified).

---

### ❌ LIMITATION

- Adds **10-20ms overhead** at 100% sampling (1-2ms at 10%)
- Requires Jaeger infrastructure monitoring (disk, queries, memory)
- At 10K+ req/day: **4-15GB/week storage** → expensive and slow
- Need to upgrade to Elasticsearch or managed APM at scale

---

### 💰 COST

**Implementation:** 4-8 hours

**Monthly:**
- <2K req/day: **$0**
- 24K req/day: **$20-50** (need Elasticsearch)
- 240K+ req/day: **$200-500** (need Tempo or managed APM)

**Alternative managed APM:** $200-1200/month

**Complexity:** 4 dependencies, 1 service, ~150 lines code, 2-4 hours/month maintenance

---

### 🤔 USE WHEN

✅ 500+ req/day with distributed system  
✅ Need to debug latency issues across services  
✅ Multiple external APIs (retrieval, reranking, LLM)  
✅ Budget >$50/month or DevOps capacity  
✅ Team can invest 1 week in setup  

---

### 🚫 AVOID WHEN

❌ <100 req/day (use structured logging)  
❌ Single-service monolith (use profilers)  
❌ <$50/month budget  
❌ MVP phase (architecture changes frequently)  
❌ >100K req/day without DevOps (use managed APM)  

In [ ]:
# Decision framework for choosing observability approach
print("Decision Framework: Should You Use Distributed Tracing?")
print("=" * 60)

def recommend_solution(req_per_day, budget_per_month, has_devops, is_distributed):
    """Recommend observability solution based on requirements."""
    
    if req_per_day < 100:
        return "Structured Logging", "Low traffic, setup time > debugging time saved"
    
    if not is_distributed:
        return "Python Profilers (py-spy, cProfile)", "Single service, no distributed context"
    
    if budget_per_month < 50:
        return "Cloud Logging", "Budget too low for self-hosted tracing"
    
    if req_per_day < 1000:
        return "Self-hosted Jaeger (50% sampling)", "Good fit for this traffic level"
    
    if req_per_day < 10000:
        return "Self-hosted Jaeger (10-20% sampling)", "Good fit for this traffic level"
    
    if req_per_day < 100000:
        if has_devops and budget_per_month < 500:
            return "OpenTelemetry + Tempo", "Cost-effective at scale with DevOps"
        else:
            return "Managed APM (Datadog, New Relic)", "Best for hands-off operation"
    
    # Very high traffic
    if has_devops:
        return "OpenTelemetry + Tempo + S3", "Scales to 1M+ req/day"
    else:
        return "Managed APM", "Operational complexity too high for self-hosted"

# Example scenarios
scenarios = [
    {"name": "MVP Startup", "req_per_day": 50, "budget": 20, "devops": False, "distributed": False},
    {"name": "Small SaaS", "req_per_day": 5000, "budget": 100, "devops": True, "distributed": True},
    {"name": "Growth Stage", "req_per_day": 50000, "budget": 300, "devops": True, "distributed": True},
    {"name": "Enterprise", "req_per_day": 500000, "budget": 2000, "devops": True, "distributed": True}
]

for scenario in scenarios:
    solution, reason = recommend_solution(
        scenario["req_per_day"],
        scenario["budget"],
        scenario["devops"],
        scenario["distributed"]
    )
    
    print(f"\n{scenario['name']}")
    print(f"  Traffic: {scenario['req_per_day']:,} req/day")
    print(f"  Budget: ${scenario['budget']}/month")
    print(f"  → Recommendation: {solution}")
    print(f"  → Reason: {reason}")

# Expected:
# MVP: Use structured logging (not tracing)
# Small SaaS: Self-hosted Jaeger (good fit)
# Growth: Tempo or managed APM
# Enterprise: Managed APM (operational simplicity)

## Summary & Next Steps

### Key Takeaways

1. **Distributed tracing provides request-level visibility** that metrics alone cannot deliver
   - Metrics: Aggregate P95 latency
   - Traces: Why THIS specific request was slow

2. **Trade-offs are real**
   - 10-20ms latency overhead (1-2ms at 10% sampling)
   - Infrastructure monitoring required (disk, queries, memory)
   - Storage scales quickly: 4-15GB/week at 10K+ req/day

3. **Not suitable for all scenarios**
   - ❌ <100 req/day → use structured logging
   - ❌ Single-service monolith → use profilers
   - ❌ MVP phase → wait for stable architecture
   - ✅ 500+ req/day, distributed system, post-MVP → tracing appropriate

4. **Alternatives exist for different scales**
   - Managed APM: Datadog, New Relic (>10K req/day, budget >$200/mo)
   - Cloud tracing: AWS X-Ray, GCP Cloud Trace (native integration)
   - Structured logging: Cost-conscious, low traffic
   - Tempo: High traffic (>100K req/day) with DevOps expertise

5. **Common failures have specific fixes**
   - Missing context propagation → use `asyncio` or explicit context attachment
   - High overhead → reduce sampling, coarser spans
   - Storage overflow → configure TTL, use Elasticsearch
   - Sampling misses traces → tail-based sampling, force-sample errors

---

### What We Built

✅ OpenTelemetry tracer with BatchSpanProcessor  
✅ Jaeger visualization backend  
✅ Full RAG pipeline instrumentation (retrieval → reranking → generation)  
✅ Trace-log correlation capability  
✅ Production-ready configuration (sampling, TTL)  

---

### Next Steps

1. **Run the FastAPI application:**
   ```bash
   python app.py
   ```

2. **Send test queries and view traces in Jaeger:**
   - Send POST to `/query` endpoint
   - Copy trace_id from response
   - Search in Jaeger UI: http://localhost:16686

3. **Run smoke tests:**
   ```bash
   pytest tests_smoke.py -v
   ```

4. **Production deployment:**
   - Configure TTL (7-30 days)
   - Set sampling rate (10-50%)
   - Monitor disk usage (<80%)
   - Set up alerts for Jaeger health

---

### Resources

- **OpenTelemetry Python Docs:** https://opentelemetry.io/docs/instrumentation/python/
- **Jaeger Documentation:** https://www.jaegertracing.io/docs/
- **Module README:** See README.md for detailed troubleshooting
- **Example Data:** example_data.json contains sample queries and failure scenarios

---

**Built with TVH Framework v2.0** - Teaching with Honesty, acknowledging trade-offs, limitations, and alternative solutions."